# 02 — BERT Outputs: `[CLS]` vs Per-Token Vectors

**Goal:** Actually run a BERT model (not just the tokenizer) and see *what comes out*. You'll understand — concretely, by looking at tensor shapes — why classification uses one output and NER uses another.

## Why this matters

When you fine-tune BERT, you're really just adding a small "head" (a linear layer) on top of its output. Different tasks use different parts of that output:

| Task | What we attach the head to |
|---|---|
| **Sequence classification** (e.g., "is this report about ransomware?") | The `[CLS]` token's output vector |
| **Token classification / NER** (e.g., "is this word a malware name?") | Every token's output vector |

Once you see the shapes with your own eyes, the rest of BERT fine-tuning is just plumbing.

## Step 1 — Load both the tokenizer and the model

In notebook 1 we only loaded the tokenizer. Now we also load the **model itself** — the neural network that produces the vectors.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()  # disables dropout; we're only doing inference here

print(f"Model loaded: {model_name}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Number of transformer layers: {model.config.num_hidden_layers}")
print(f"Number of attention heads: {model.config.num_attention_heads}")

Model loaded: bert-base-uncased
Hidden size: 768
Number of transformer layers: 12
Number of attention heads: 12


### What you just loaded

- **Hidden size 768** — every token will be represented by a 768-dimensional vector.
- **12 transformer layers** — BERT has 12 stacked transformer blocks. Each one refines the token representations.
- **12 attention heads** — within each layer, 12 parallel "views" of how tokens relate to each other.

> `AutoModel` gives you BERT *without any task-specific head*. It returns raw hidden states. Later we'll use `AutoModelForTokenClassification` (has an NER head) and `AutoModelForSequenceClassification` (has a classification head).

## Step 2 — Run BERT on a CTI sentence

We'll push our CTI sentence through BERT and inspect the output tensors.

In [2]:
cti_sentence = "APT28 deployed X-Agent malware exploiting CVE-2017-0199."

inputs = tokenizer(cti_sentence, return_tensors="pt")
print("input_ids shape:", inputs["input_ids"].shape)
print("input_ids:", inputs["input_ids"])

with torch.no_grad():  # disable gradient tracking for inference (faster, less memory)
    outputs = model(**inputs)

print("\nOutput keys:", outputs.keys())

input_ids shape: torch.Size([1, 21])
input_ids: tensor([[  101, 26794, 22407,  7333,  1060,  1011,  4005, 15451,  8059, 18077,
          2075, 26226,  2063,  1011,  2418,  1011,  5890,  2683,  2683,  1012,
           102]])

Output keys: odict_keys(['last_hidden_state', 'pooler_output'])


BERT returned two things: `last_hidden_state` and `pooler_output`. Let's look at their shapes — this is the key insight of the whole notebook.

In [3]:
print("last_hidden_state shape:", outputs.last_hidden_state.shape)
print("pooler_output shape:   ", outputs.pooler_output.shape)

last_hidden_state shape: torch.Size([1, 21, 768])
pooler_output shape:    torch.Size([1, 768])


### Decoding the shapes

Suppose your input had **16 tokens** (including `[CLS]` and `[SEP]`).

- `last_hidden_state` → `[1, 16, 768]`
  - `1` = batch size (one sentence)
  - `16` = one vector **per token**
  - `768` = each vector has 768 dimensions
  - **→ This is what NER uses.** One label prediction per token.

- `pooler_output` → `[1, 768]`
  - `1` = batch size
  - `768` = one vector for the **whole sentence** (derived from the `[CLS]` token)
  - **→ This is what classification uses.** One label prediction per sentence.

That's it. That's the whole difference between NER and classification at the output level.

## Step 3 — Extract the `[CLS]` vector (for classification)

The `[CLS]` token is always at **position 0**. Its hidden state is the sentence-level representation.

In [4]:
cls_vector = outputs.last_hidden_state[0, 0, :]  # [batch=0, token=0, all 768 dims]

print("CLS vector shape:", cls_vector.shape)
print("First 10 values:", cls_vector[:10].tolist())
print(f"\nThis 768-dim vector represents the whole sentence: '{cti_sentence}'")

CLS vector shape: torch.Size([768])
First 10 values: [-1.1891640424728394, -0.21495720744132996, 0.23942413926124573, 0.16603229939937592, -0.15723495185375214, -0.2069702297449112, 0.19654002785682678, 0.3451342284679413, -0.12225811183452606, -0.4033951759338379]

This 768-dim vector represents the whole sentence: 'APT28 deployed X-Agent malware exploiting CVE-2017-0199.'


**For fine-tuning a classifier**, we'd add a linear layer like `nn.Linear(768, num_classes)` on top of this vector. That's exactly what `BertForSequenceClassification` does internally.

## Step 4 — Extract per-token vectors (for NER)

For NER we need the hidden state at *every* token position so we can predict a label for each one.

In [5]:
token_ids = inputs["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(token_ids)
token_vectors = outputs.last_hidden_state[0]  # [num_tokens, 768]

print(f"Got {token_vectors.shape[0]} token vectors, each of size {token_vectors.shape[1]}\n")

print(f"{'Pos':<4} {'Token':<15} {'First 5 dims of its vector'}")
print("-" * 70)
for i, (tok, vec) in enumerate(zip(tokens, token_vectors)):
    preview = [round(v, 3) for v in vec[:5].tolist()]
    print(f"{i:<4} {tok:<15} {preview}")

Got 21 token vectors, each of size 768

Pos  Token           First 5 dims of its vector
----------------------------------------------------------------------
0    [CLS]           [-1.189, -0.215, 0.239, 0.166, -0.157]
1    apt             [0.159, -0.714, 0.241, 0.203, -0.151]
2    ##28            [-0.428, -0.398, 0.469, 0.256, 0.663]
3    deployed        [-1.134, 0.135, 0.078, 0.264, 0.459]
4    x               [0.013, 0.143, 0.514, -0.133, 0.637]
5    -               [-1.225, -0.54, 0.186, 0.119, 0.427]
6    agent           [-0.57, -0.635, 0.097, -0.187, 0.037]
7    mal             [0.606, 0.027, 0.213, 0.197, 0.84]
8    ##ware          [-0.539, -0.483, 0.078, 0.228, 1.355]
9    exploit         [-1.057, 0.223, -0.103, 0.085, 0.28]
10   ##ing           [-1.831, -0.005, 0.109, 0.509, 0.363]
11   cv              [0.089, -0.855, 1.108, 0.383, -0.321]
12   ##e             [-0.737, -0.219, 0.371, 0.425, 0.496]
13   -               [-0.386, -0.455, 0.173, 0.324, 0.663]
14   2017            

**For fine-tuning an NER model**, we'd add `nn.Linear(768, num_entity_labels)` and apply it to *every* token vector. That's what `BertForTokenClassification` does.

Notice that `apt` and `##28` each have their own 768-dim vector. When we do NER, we'll need to decide:
- Assign a label to every subword?
- Only the first subword, and ignore the rest?
- Only score loss on the first subword?

This is the **subword-label alignment** problem. We'll tackle it head-on in notebook 4.

## Step 5 — Sanity check: similar sentences produce similar `[CLS]` vectors

If `[CLS]` really captures sentence meaning, then two related CTI sentences should have more similar `[CLS]` vectors than two unrelated ones. Let's verify.

> **Caveat:** raw `bert-base-uncased` is *not* great at semantic similarity (that's what Sentence-BERT is for), but we should still see some signal.

In [6]:
import torch.nn.functional as F

def get_cls_vector(text):
    inp = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = model(**inp)
    return out.last_hidden_state[0, 0, :]

sentences = [
    "APT28 deployed X-Agent malware.",
    "Fancy Bear used their custom malware in the attack.",  # semantically related
    "I baked a chocolate cake this morning.",               # unrelated
]

vecs = [get_cls_vector(s) for s in sentences]

def cosine(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print(f"sim(CTI-1, CTI-2):     {cosine(vecs[0], vecs[1]):.4f}  <- should be highest")
print(f"sim(CTI-1, cake):      {cosine(vecs[0], vecs[2]):.4f}")
print(f"sim(CTI-2, cake):      {cosine(vecs[1], vecs[2]):.4f}")

sim(CTI-1, CTI-2):     0.8352  <- should be highest
sim(CTI-1, cake):      0.7975
sim(CTI-2, cake):      0.8372


The two CTI sentences should show higher similarity than either does to the cake sentence. The differences won't be huge (raw BERT isn't ideal for similarity), but the ordering should be right.

## Step 6 — Batching: real fine-tuning always uses batches

Running one sentence at a time is inefficient. Let's push 3 sentences through BERT at once. **This is where `attention_mask` becomes essential.**

In [7]:
batch = [
    "APT28 deployed X-Agent.",
    "Emotet is a banking trojan that spreads via phishing emails with macro-enabled documents.",
    "The threat actor exploited CVE-2021-44228.",
]

# padding=True -> pad shorter sentences to match the longest one
# truncation=True -> cut anything over model_max_length
batch_inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")

print("input_ids shape:     ", batch_inputs["input_ids"].shape)
print("attention_mask shape:", batch_inputs["attention_mask"].shape)
print()
print("Attention masks (1 = real token, 0 = padding):")
print(batch_inputs["attention_mask"])

input_ids shape:      torch.Size([3, 21])
attention_mask shape: torch.Size([3, 21])

Attention masks (1 = real token, 0 = padding):
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]])


In [8]:
with torch.no_grad():
    batch_outputs = model(**batch_inputs)

print("Batched last_hidden_state shape:", batch_outputs.last_hidden_state.shape)
# -> [batch_size, max_seq_len_in_batch, 768]

# One [CLS] vector per sentence — ready for a classification head
cls_batch = batch_outputs.last_hidden_state[:, 0, :]
print("CLS vectors for the batch:", cls_batch.shape)

Batched last_hidden_state shape: torch.Size([3, 21, 768])
CLS vectors for the batch: torch.Size([3, 768])


### Why `attention_mask` matters

The short sentences were padded with `[PAD]` tokens so the batch is a rectangle. BERT uses the attention mask to **ignore** padded positions — they don't influence the real tokens' representations.

If you forgot the mask, BERT would attend to meaningless padding and your outputs would be subtly wrong.

## Step 7 — Preview: what the task-specific models add

Let's very briefly load the two task-specific wrappers we'll use later, just to see the head that's been bolted on top.

In [12]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification

# Classification head: one Linear(768 -> num_labels) on the [CLS] vector
clf_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3  # e.g., 3 CTI report categories
)
print("Classification head:")
print(clf_model.classifier)
print()

# Token classification head: one Linear(768 -> num_labels) applied to EVERY token
ner_model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=9  # e.g., 9 BIO labels for 4 entity types
)
print("NER head:")
print(ner_model.classifier)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Classification head:
Linear(in_features=768, out_features=3, bias=True)

NER head:
Linear(in_features=768, out_features=9, bias=True)


You'll see a warning like *"Some weights of the model were not initialized..."*. That's expected — the classification head is **randomly initialized** because we're about to train it from scratch on our specific task.

**The head is tiny** — just a single `Linear` layer in each case. Almost all the intelligence is in the 110M pretrained BERT parameters underneath.

## Your turn — exercises

1. Compute the **mean of all token vectors** (excluding `[CLS]` and `[SEP]`) for a CTI sentence. Compare its cosine similarity to `[CLS]`. Are they interchangeable as sentence representations?
2. Run a very long CTI sentence (~700 words) through BERT. What error do you get? How does `truncation=True` fix it?
3. Load `AutoModelForSequenceClassification` with `num_labels=5` and inspect `clf_model.classifier.out_features`. Confirm the head adapts to the task.

In [13]:
# Your experiments here

## Summary

- `AutoModel` returns `last_hidden_state` of shape `[batch, seq_len, 768]` — one vector per token.
- The vector at position 0 is the `[CLS]` vector — used for **sentence classification**.
- Each other position gives us a token vector — used for **NER**.
- In batches, `attention_mask` tells BERT which positions are real vs. padded.
- Task-specific models (`AutoModelForSequenceClassification`, `AutoModelForTokenClassification`) just add a small `Linear` head on top of these vectors.

## Next notebook

**`03_bio_tagging_for_ner.ipynb`** — before we can fine-tune for NER, we need to understand how labels are represented. We'll cover the **BIO tagging scheme** on a real CTI sentence, manually tag entities (threat actor, malware, CVE), and see why BIO is the dominant labeling format for NER.